# Verify HU and Plaque Context

Clean Colab workflow for testing whether nnU-Net plaque labels are mostly calcified cores and whether nearby dilated-shell voxels contain lower-attenuation, noncalcified plaque-like tissue.

Research use only. Not clinically validated. Not for diagnosis.

## 1. Install and clone/pull OpenPlaque

In [ ]:
from pathlib import Path
import inspect, os, sys, subprocess

REPO_URL = "https://github.com/pazzani/OpenPlaque.git"
REPO_DIR = Path("/content/OpenPlaque")
REPO_BRANCH = "agent/plaque-context-verification"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-colab.txt")], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", "-e", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))

import openplaque
import openplaque.plaque_context as plaque_context
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
branch = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
signature = inspect.signature(plaque_context.compute_reports_plaque_context)
assert "vessel_label" in signature.parameters, f"Unexpected plaque_context version from {plaque_context.__file__}: {signature}"
print("Using", REPO_DIR)
print("Branch", branch)
print("Commit", commit)
print("openplaque from", openplaque.__file__)
print("plaque_context signature", signature)


## 2. Configure UCLA study, model, and outputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
STUDY_ZIP = DRIVE_ROOT / 'Full_DICOM.zip'
MODEL_ZIP = DRIVE_ROOT / 'models' / 'Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'
OUTPUT_DIR = DRIVE_ROOT / 'UCLA_Plaque_Context_Verification'
MASK_DIR = OUTPUT_DIR / 'nnunet_masks'
PLOT_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

FALLBACK_SERIES = {'RCA': 1035, 'LCX': 1039, 'LAD': 1043}
VESSELS = ['LAD', 'RCA', 'LCX']
REUSE_SEGMENTATIONS = True
RUN_NNUNET_IF_MISSING = True
RADII_VOXELS = (1, 2, 3)
VESSEL_DILATION_VOXELS = 1
CONTEXT_MIN_HU = 30
CONTEXT_MAX_HU = 350

print('Study:', STUDY_ZIP)
print('Outputs:', OUTPUT_DIR)


## 3. Install or expose nnU-Net model

In [ ]:
import zipfile, shutil

NNUNET_RESULTS = Path('/content/nnUNet_results')
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ.setdefault('nnUNet_raw', '/content/nnUNet_raw')
os.environ.setdefault('nnUNet_preprocessed', '/content/nnUNet_preprocessed')

if MODEL_ZIP.exists() and not (NNUNET_RESULTS / 'Dataset001_CCTA_DHM').exists():
    NNUNET_RESULTS.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(MODEL_ZIP) as zf:
        zf.extractall(NNUNET_RESULTS)

print('nnUNet_results =', os.environ['nnUNet_results'])

## 4. Load UCLA DICOM study and detect LAD/RCA/LCX

In [ ]:
from openplaque.study import OpenPlaqueStudy
from openplaque.run_new_data import auto_detect_or_fallback_series

study = OpenPlaqueStudy(str(STUDY_ZIP))
series_map = auto_detect_or_fallback_series(study, fallback_series=FALLBACK_SERIES)
series_map = {k: int(series_map[k]) for k in VESSELS}
series_map

## 5. Run or reuse nnU-Net segmentations

In [ ]:
import numpy as np
import SimpleITK as sitk
from openplaque.segmentation import SegmentationReport, export_for_nnunet, run_nnunet, load_segmentation

reports = []
for vessel in VESSELS:
    image, volume, _ = study.load_series(series_map[vessel])
    saved_mask = MASK_DIR / f'{vessel}.nii.gz'
    if REUSE_SEGMENTATIONS and saved_mask.exists():
        mask_image = sitk.ReadImage(str(saved_mask))
        mask = sitk.GetArrayFromImage(mask_image)
        input_dir = str(OUTPUT_DIR / f'{vessel}_input')
        output_dir = str(MASK_DIR)
        print(f'Reused {saved_mask}')
    elif RUN_NNUNET_IF_MISSING:
        input_dir = str(OUTPUT_DIR / f'{vessel}_input')
        output_dir = str(OUTPUT_DIR / f'{vessel}_nnunet_output')
        export_for_nnunet(image, input_dir, vessel)
        run_nnunet(input_dir, output_dir)
        mask_image, mask = load_segmentation(output_dir, vessel)
        sitk.WriteImage(mask_image, str(saved_mask))
        print(f'Ran nnU-Net and saved {saved_mask}')
    else:
        raise FileNotFoundError(f'Missing saved mask for {vessel}: {saved_mask}')

    reports.append(SegmentationReport(vessel, image, mask_image, volume, mask, input_dir, output_dir))

[(r.name, r.label_counts(), r.hu_statistics()) for r in reports]

## 6. Compute plaque-core and shell HU summaries

In [ ]:
import importlib
import pandas as pd
import openplaque.plaque_context as plaque_context
plaque_context = importlib.reload(plaque_context)
compute_reports_plaque_context = plaque_context.compute_reports_plaque_context
save_context_csv = plaque_context.save_context_csv
write_context_html_report = plaque_context.write_context_html_report

geometric_rows = compute_reports_plaque_context(
    reports,
    radii_voxels=RADII_VOXELS,
    label=2,
    connectivity=26,
)
for row in geometric_rows:
    row['context_mode'] = 'geometric_shell'

filtered_rows = compute_reports_plaque_context(
    reports,
    radii_voxels=RADII_VOXELS,
    label=2,
    vessel_label=1,
    connectivity=26,
    anatomical_filter=True,
    vessel_dilation_voxels=VESSEL_DILATION_VOXELS,
    candidate_min_hu=CONTEXT_MIN_HU,
    candidate_max_hu=CONTEXT_MAX_HU,
    include_candidate_rows=True,
)
for row in filtered_rows:
    row['context_mode'] = 'anatomy_filtered'

rows = geometric_rows + filtered_rows
df = pd.DataFrame(rows)
csv_path = save_context_csv(rows, OUTPUT_DIR / 'plaque_hu_context_summary.csv')
summary_cols = ['context_mode', 'vessel', 'region', 'voxel_count', 'mean_hu', 'hu_30_130_fraction', 'hu_130_350_fraction', 'hu_350_700_fraction', 'hu_700_1000_fraction', 'gt_1000_fraction']
df[summary_cols].sort_values(['context_mode', 'vessel', 'radius_voxels', 'region'])


## 7. Generate histograms and overlays

In [ ]:
import importlib
import openplaque.plaque_context as plaque_context
plaque_context = importlib.reload(plaque_context)
save_hu_histogram_png = plaque_context.save_hu_histogram_png
save_context_overlay_png = plaque_context.save_context_overlay_png
vessel_context_mask = plaque_context.vessel_context_mask

histogram_paths = []
overlay_paths = []
for report in reports:
    include_mask = vessel_context_mask(
        report.mask,
        vessel_label=1,
        plaque_label=2,
        vessel_dilation_voxels=VESSEL_DILATION_VOXELS,
        connectivity=26,
    )
    histogram_paths.append(save_hu_histogram_png(
        report.volume,
        report.mask,
        PLOT_DIR / f'{report.name}_anatomy_filtered_context_histogram.png',
        f'{report.name} anatomy-filtered plaque context HU',
        radii_voxels=RADII_VOXELS,
        include_mask=include_mask,
    ))
    overlay_paths.append(save_context_overlay_png(
        report.volume,
        report.mask,
        PLOT_DIR / f'{report.name}_context_candidate_overlay.png',
        f'{report.name} candidate context: {CONTEXT_MIN_HU}-{CONTEXT_MAX_HU} HU',
        radius_voxels=3,
        include_mask=include_mask,
        candidate_min_hu=CONTEXT_MIN_HU,
        candidate_max_hu=CONTEXT_MAX_HU,
    ))

html_path = write_context_html_report(
    OUTPUT_DIR / 'plaque_hu_context_report.html',
    rows,
    histogram_paths=histogram_paths,
    overlay_paths=overlay_paths,
)
print('CSV:', csv_path)
print('HTML:', html_path)
print('Plots:', PLOT_DIR)


## 8. Quick interpretation table

High plaque-core fractions in `hu_350_700`, `hu_700_1000`, and `gt_1000`, combined with higher shell fractions in `hu_30_130` or `hu_130_350`, support the hypothesis that the nnU-Net label is focused on calcified plaque cores while surrounding voxels may include noncalcified or mixed plaque context.

In [ ]:
display_cols = [
    'context_mode', 'vessel', 'region', 'voxel_count', 'mean_hu', 'median_hu',
    'lt_30_fraction', 'hu_30_130_fraction', 'hu_130_350_fraction',
    'hu_350_700_fraction', 'hu_700_1000_fraction', 'gt_1000_fraction',
]
df.sort_values(['context_mode', 'vessel', 'radius_voxels', 'region'])[display_cols]
